# Workshop exercises: single-experiment inference

This participant copy keeps the single-experiment walkthrough intact, then adds guided experiments. First run the notebook once with its defaults (`s000004`, `PRIORS_DEV = 30`). Use the exercise section at the end to make one change at a time and compare the outputs. The walkthrough notebook remains unchanged.


## 1. Setup

Add the project root to `sys.path` so the pipeline modules are importable from within the `notebooks/` subdirectory.

In [ ]:
%matplotlib inline

import sys
import os

# Add project root to path when running from notebooks/ subdirectory
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
from device_utils import detect_torch_device, summarize_torch_backends

backend_summary = summarize_torch_backends()
device = detect_torch_device()
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA (NVIDIA GPU) available  : {backend_summary['cuda']}")
print(f"MPS (Apple Silicon) available   : {backend_summary['mps']}")
print(f"Device          : {device}")
print(f"Project root    : {PROJECT_ROOT}")


## 2. Choose an Experiment

Set `EXPERIMENT_ID` to a sample identifier in `sXXXXXX` format (e.g. `"s000004"`).
Available IDs can be found by listing `dataset/test/`.
`LAYER_COUNT` describes the film layer structure used for prior construction and parameter parsing.

In [ ]:
# fixed parameters for this notebook
LAYER_COUNT   = 1           # picking an experiment with different layer counts will require different model (NF_CONFIG)
NF_CONFIG     = "example_nf_config_reflectorch.yaml"

In [ ]:
EXPERIMENT_ID = "s000004" # any experiment from 'dataset/test/'

In [ ]:
# Parameters for the experiment
NF_SAMPLES    = 1000
PRIORS_DEV    = 30      # prior width as % of the model constraint range

# Preprocessing parameters
ENABLE_PREPROCESSING = True
ERROR_THRESHOLD      = 0.75
CONSECUTIVE          = 5

## 3. Discover Experiment Files

`parameter_discovery.discover_experiment_files` locates the reflectivity data file and the ground-truth model file for the chosen experiment.

In [ ]:
from pathlib import Path
from config import DATA_DIRECTORY
from parameter_discovery import parse_true_parameters_from_model_file

data_dir   = Path(PROJECT_ROOT) / DATA_DIRECTORY
data_file  = data_dir / f"{EXPERIMENT_ID}_experimental_curve.dat"
model_file = data_dir / f"{EXPERIMENT_ID}_model.txt"

print(f"Data file (measured curve) : {data_file}")
print(f"Model file (ground truth parameters): {model_file}")

# Parse true structural parameters from the model file
true_params_dict = parse_true_parameters_from_model_file(model_file)

layer_key   = f"{LAYER_COUNT}_layer"
param_names = true_params_dict[layer_key]["param_names"]
true_params  = true_params_dict[layer_key]["params"]

print(f"\nTrue parameters ({LAYER_COUNT}-layer):")
for name, val in zip(param_names, true_params):
    print(f"  {name:<20} = {val:.4f}")


## 4. Load and Preprocess Experimental Data

The preprocessing step truncates the Q-range at the first run of consecutive high relative-error points.
This prevents tensor concatenation errors during NF inference.

In [ ]:
from simple_pipeline import load_experimental_data


q_exp, curve_exp, sigmas_exp = load_experimental_data(
    data_file_path=data_file,
    enable_preprocessing=ENABLE_PREPROCESSING,
    threshold=ERROR_THRESHOLD,
    consecutive=CONSECUTIVE,
)


## 5. Build Constraint-Based Prior Bounds

The prior bounds are constructed with width `PRIORS_DEV`% of each model constraint range, centred on the true parameter.
The window is additionally clipped to the physical constraint ranges defined in `model_constraints.json`.

In [ ]:
from parameter_discovery import get_prior_bounds_for_experiment

prior_bounds = get_prior_bounds_for_experiment(
    experiment_id=EXPERIMENT_ID,
    true_params_dict=true_params_dict,
    layer_count=LAYER_COUNT,
    priors_type="constraint_based",
    deviation=PRIORS_DEV / 100.0,
)

print("Prior bounds (constraint-based, deviation = {:.0%}):".format(PRIORS_DEV / 100.0))
print(f"{'Parameter':<20}  {'Min':>10}  {'True':>10}  {'Max':>10}")
print("-" * 56)
for name, val, (lo, hi) in zip(param_names, true_params, prior_bounds):
    print(f"{name:<20}  {lo:>10.4f}  {val:>10.4f}  {hi:>10.4f}")


## 6. Load Model and Run Inference

`EasyInferenceModel` loads the trained NF weights and config. `preprocess_and_sample` interpolates the data onto the model Q-grid and draws `NF_SAMPLES` posterior samples via importance-weighted sampling.

In [ ]:
from reflectorch import EasyInferenceModel
from simple_pipeline import _extend_prior_bounds_for_nf
from device_utils import detect_torch_device

torch.manual_seed(420)
device = detect_torch_device()
inference_model = EasyInferenceModel(
    config_name=NF_CONFIG,
    device=device,
)
print(f"Using inference device: {device}")

# Interpolate onto the model Q-grid (includes sigmas for dR-aware models)
q_model, curve_interp, sigmas_interp = inference_model.interpolate_data_to_model_q(
    q_exp, curve_exp, sigmas_exp=sigmas_exp
)

# Extend prior bounds with nuisance parameters (r_scale, log10_background)
# required by the NF model, handled automatically by _extend_prior_bounds_for_nf
prior_bounds_nf = _extend_prior_bounds_for_nf(prior_bounds)

prediction_dict = inference_model.preprocess_and_sample(
    reflectivity_curve=curve_interp,
    q_values=q_model,
    sigmas=sigmas_interp,
    ambient_sld=float(np.loadtxt(model_file, comments="#", usecols=1, max_rows=1) * 1e6),
    num_samples=NF_SAMPLES,
    prior_bounds=prior_bounds_nf,
    q_resolution=0.1,
    clip_prediction=True,
    enable_error_bars_filtering=False,  # q_exp was already preprocessed above
    calc_sampled_curves=True,
    calc_sampled_sld_profiles=True,
    calc_log_likelihoods=True,
    enable_importance_sampling=True,
)

print("Inference complete.")
print(f"Sampled params shape : {prediction_dict['predicted_params_array'].shape}")
print(f"Parameter names      : {prediction_dict['param_names']}")


## 7. Extract MAP Estimate

Select the highest log-likelihood sample as the MAP (maximum a posteriori) estimate.

In [ ]:
from nf_statistics import compute_nf_sample_statistics

log_likelihoods = prediction_dict["log_likelihoods"]
pred_params     = prediction_dict["predicted_params_array"]
nf_param_names  = prediction_dict["param_names"]

best_idx   = int(np.argmax(log_likelihoods))
map_params = pred_params[best_idx]

print(f"MAP sample index     : {best_idx}")
print(f"MAP log-likelihood   : {log_likelihoods[best_idx]:.4f}")
print()

stats = compute_nf_sample_statistics(pred_params, log_likelihoods)

print(f"{'Parameter':<20}  {'True':>10}  {'MAP':>10}  {'Mean':>10}  {'Std':>10}")
print("-" * 66)
for i, name in enumerate(nf_param_names):
    true_val = true_params[i] if i < len(true_params) else float("nan")
    print(
        f"{name:<20}  {true_val:>10.4f}  {map_params[i]:>10.4f}"
        f"  {stats['nf_params_mean'][i]:>10.4f}  {stats['nf_params_std'][i]:>10.4f}"
    )


## 8. Visualise Reflectivity Curve Fit and SLD Profile

`plot_simple_comparison` produces a two-panel figure: the experimental curve with the MAP fit overlaid, and the corresponding SLD depth profile.

In [ ]:
from plotting_utils import plot_simple_comparison
from parameter_discovery import generate_true_sld_profile

# Disable LaTeX rendering for better compatibility
plt.rcParams["text.usetex"] = False

predicted_sld_x = np.asarray(prediction_dict["sampled_sld_xaxis"])
fronting_sld = float(np.loadtxt(model_file, comments="#", usecols=1, max_rows=1) * 1e6)
true_sld_x, true_sld_y = generate_true_sld_profile(
    true_params_dict, ambient_sld=fronting_sld, x_axis=predicted_sld_x
)

plot_simple_comparison(
    q_exp=q_exp,
    curve_exp=curve_exp,
    sigmas_exp=sigmas_exp,
    q_model=np.asarray(prediction_dict["q_plot_pred"]),
    predicted_curve=np.asarray(prediction_dict["sampled_curves"])[best_idx],
    polished_curve=np.asarray(prediction_dict["sampled_curves"])[best_idx],
    predicted_sld_x=predicted_sld_x,
    predicted_sld_y=np.asarray(prediction_dict["sampled_sld_profiles"])[best_idx],
    polished_sld_y=np.asarray(prediction_dict["sampled_sld_profiles"])[best_idx],
    true_sld_x=true_sld_x,
    true_sld_y=true_sld_y,
    experiment_name=EXPERIMENT_ID,
)


## 9. Compute Constraint MAPE Metrics

Constraint MAPE normalises the absolute prediction error by the prior half-width rather than the true value:

$$\text{Constraint MAPE} = \frac{|\hat{y} - y|}{\text{constraint width}} \times 100$$

A value of 0 means the prediction equals the true value; 100 means the prediction is at the constraint boundary.

In [ ]:
from error_calculation import calculate_parameter_metrics, print_metrics_report

# Slice to structural params only (exclude nuisance: r_scale, log10_background).
# Use `param_names` (standardized keys from parameter_discovery) rather than
# `nf_param_names` (human-readable NF labels) so get_constraint_width can look
# them up in model_constraints.json.
n_structural = len(true_params)
map_params_structural = map_params[:n_structural]

param_metrics = calculate_parameter_metrics(
    pred_params=map_params_structural,
    true_params=true_params,
    param_names=param_names,
    prior_bounds=prior_bounds,
    priors_type="constraint_based",
)

print_metrics_report(param_metrics=param_metrics)


## Hands-on exercises: priors, posterior samples, and SLD profiles

The walkthrough plots only the highest-log-likelihood sample. That is useful for a quick fit, but it can hide other plausible solutions. Compare the prior bounds, parameter samples, reflectivity fit, and sampled SLD profiles.

> **Teaching setup:** the constraint-based prior in this notebook is centered on known ground truth. This makes prior-width effects easy to demonstrate; a real unknown sample would not provide the true parameters for centering its prior.

### 1. Predict, then vary the prior width

Keep `EXPERIMENT_ID = "s000004"`. Before running inference, use the true thickness (28.63 Å) and the thickness constraint span (999 Å) to predict the bounds. `PRIORS_DEV` sets the total interval width to `PRIORS_DEV / 100 × 999 Å`; the half-width is half of that. Bounds are shifted or clipped to stay inside 1–1000 Å. Fill in the predicted ranges, then compare them with the printed bounds.

| `PRIORS_DEV` | Predicted thickness bounds (Å) | Observed MAP thickness (Å) | SLD profile changed? What changed? |
| ---: | --- | ---: | --- |
| 5 | | | |
| 30 (baseline) | | | |
| 60 | | | |

After each change, rebuild `prior_bounds` and rerun inference and the result cells. What changes when the prior expands: only the uncertainty, or the best-fit structure too?

### 2. Choose a different experiment

Choose at least two rows and set `EXPERIMENT_ID` to each sample in turn. Rerun from experiment discovery and data loading through inference, plots, and metrics. Keep `LAYER_COUNT = 1` and the existing NF config. These cases are selected to vary thickness, Q coverage, SLD, and roughness; the suggested effects are questions to investigate, not promised outcomes.

| Experiment | True thickness | Layer SLD (10⁻⁶ Å⁻²) | Layer roughness | Raw Qmax / points | After preprocessing: Qmax / points | Experiment idea |
| --- | ---: | ---: | ---: | ---: | ---: | --- |
| `s000004` | 28.63 Å | 3.567 | 7.04 Å | 0.284 Å⁻¹ / 76 | 0.174 Å⁻¹ / 67 | Baseline; compare the narrow and broad priors. |
| `s001605` | 727.95 Å | 4.291 | 28.06 Å | 0.233 Å⁻¹ / 96 | 0.150 Å⁻¹ / 91 | Thick layer and lowest retained Q reach; compare 5%, 30%, and 60% widths. |
| `s008778` | 778.57 Å | 8.178 | 15.80 Å | 0.258 Å⁻¹ / 104 | 0.190 Å⁻¹ / 99 | Compare a second thick layer with `s001605`: do wider priors change the preferred thickness or SLD? |
| `s007440` | 185.48 Å | 5.072 | 3.42 Å | 0.258 Å⁻¹ / 79 | 0.258 Å⁻¹ / 79 | Sparser curve and smoother interface; compare with a thicker sample. |
| `s004818` | 447.73 Å | 10.813 | 6.12 Å | 0.258 Å⁻¹ / 105 | 0.165 Å⁻¹ / 96 | High layer SLD; inspect the profile samples and their spread. |
| `s003916` | 389.52 Å | 8.489 | 43.09 Å | 0.258 Å⁻¹ / 86 | 0.258 Å⁻¹ / 86 | High layer roughness; check whether thickness and roughness trade off. |

For `s001605`, the clipped thickness bounds are about 703–753 Å at 5%, 578–878 Å at 30%, and 401–1000 Å at 60%. Do the wider bounds actually produce another thickness solution or SLD profile? This is a hypothesis to test, not a guaranteed result.

### 3. Check whether the posterior has more than one mode (optional)

The main plot selects the highest-log-likelihood sample. For a posterior histogram, use the importance weights returned by the model; `pred_params` are the proposal samples, not equally weighted posterior draws. Fill in the blanks in a new code cell. `param_names` contains structural parameter names, and the structural samples are at the start of `pred_params`.

```python
thickness_idx = param_names.index("________")
plt.hist(pred_params[:, thickness_idx], bins=____, weights=prediction_dict["weights"], density=True)
plt.xlabel("Thickness (Å)")
plt.show()
```

Also report `prediction_dict["ess"]` (effective sample size) relative to `len(pred_params)`. If ESS is very low, the weights are concentrated on too few samples: do not claim posterior modes from that run. Increasing `NF_SAMPLES` may help, but check whether ESS actually improves. If the weighted histogram suggests separated groups, compare high-weight samples from each group by plotting their entries in `prediction_dict["sampled_sld_profiles"]` against `prediction_dict["sampled_sld_xaxis"]`. Do distinct thickness groups produce different SLD profiles, or nearly equivalent fits?

**Interpretation check from the instructor run:** with this checkpoint and importance-sampling setting, the six 1,000-sample runs had ESS 1.0–1.7, and the 10,000-sample `s001605` pilot still had ESS 1.0. The highest-log-likelihood SLD profile changes with prior width, but these runs do not establish distinct posterior modes. Treat the low ESS as part of the experiment result.

| Run | Experiment | Prior width | Number/location of thickness peaks | SLD-profile observation |
| --- | --- | ---: | --- | --- |
| Baseline | s000004 | 30% | | |
| Narrow prior | | 5% | | |
| Wide prior | | 60% | | |
| Chosen experiment | | | | |
